# Energy Load Forecasting
### Predicting Hourly Electricity Demand for Spain using ENTSO-E Data

**Project 17 of 50 — Time Series Forecasting Portfolio**

## Project Overview

Spain's electricity grid operator publishes hourly data on energy generation, consumption, and prices via ENTSO-E. This notebook forecasts **total hourly electricity load (MW)** — a critical task for grid operators who must balance supply and demand in real time.

We use the Kaggle dataset 
icholasjhana/energy-consumption-generation-prices-and-weather which contains 4 years of hourly observations (2015-2018) with 29 columns including generation by source (solar, wind, nuclear, hydro, gas), actual and forecast load, and spot prices.

| Attribute | Value |
|---|---|
| **Dataset** | ENTSO-E Spain Energy (Kaggle) |
| **Target** | 	otal load actual (MW) |
| **Date column** | 	ime |
| **Frequency** | Hourly |
| **TS Library** | Darts |
| **Season length** | 24 h (daily), 168 h (weekly) |

## Learning Objectives

1. Load and clean a real-world multi-column hourly energy dataset
2. Understand multi-scale seasonality: daily (24h) and weekly (168h) patterns
3. Build time-aware train/validation/test splits on hourly data
4. Engineer lag features specific to energy load (hour-of-day, day-of-week, temperature proxy)
5. Benchmark with LazyPredict on tabular lag features
6. Run FLAML AutoML for regression on lag features
7. Use **Darts** to fit ExponentialSmoothing, AutoARIMA, and NaiveSeasonal models
8. Select top 3 models by RMSE on the held-out test set
9. Perform thorough error analysis (bias, intra-day error patterns)
10. Discuss grid-operations implications of forecast accuracy

## Problem Statement

Given 4 years of hourly electricity load observations for the Spanish grid, forecast the next **168 hours (1 week)** of total load.

This is a univariate forecasting problem with strong dual seasonality. Residual errors from an inaccurate forecast translate directly into balancing costs for grid operators.

## Why This Project Matters

Electricity cannot be stored at scale. Grid operators must procure exactly the right amount of generation capacity 24–48 hours ahead. A 1% forecast error on a system like Spain's (~40 GW peak) represents ~400 MW — equivalent to operating a mid-size gas plant unnecessarily or risking blackouts.

Accurate load forecasting reduces:
- Reserve margin costs (over-procurement of backup capacity)
- Balancing market costs (real-time corrections)
- CO₂ emissions (unnecessary fossil dispatch)

## Dataset Overview

**Source:** Kaggle — 
icholasjhana/energy-consumption-generation-prices-and-weather

**Key columns:**
| Column | Description |
|---|---|
| 	ime | Hourly timestamp (UTC+1 Spain) |
| 	otal load actual | **Target** — actual hourly grid load in MW |
| 	otal load forecast | ENTSO-E official day-ahead forecast (upper baseline) |
| price actual | Day-ahead spot price €/MWh |
| generation solar | Solar generation MW |
| generation wind onshore | Wind generation MW |
| generation fossil gas | Gas generation MW |
| 	emp, humidity etc. | Weather covariates (5 Spanish cities) |

**Period:** 2015-01-01 to 2018-12-31 (~35,000 rows)
**License:** Open Data — ENTSO-E / AEMET / OMIE

## Dataset Source & License

- **Kaggle slug:** 
icholasjhana/energy-consumption-generation-prices-and-weather
- **Upstream sources:** ENTSO-E (load/generation), OMIE (prices), AEMET (weather) — all open data
- **License:** CC BY 4.0 / Open Data
- **Educational use only:** Do not use forecasts from this notebook for actual grid operations

## Environment Setup

Install required libraries. Run once per environment.

In [ ]:
import subprocess, sys

pkgs = ["kagglehub", "pandas", "numpy", "matplotlib", "seaborn",
        "plotly", "scikit-learn", "lazypredict", "flaml",
        "darts", "statsmodels", "scipy"]

for pkg in pkgs:
    try:
        __import__(pkg.split("[")[0].replace("-","_"))
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

print("Environment ready.")

## Imports

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import os, pathlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.linear_model import Ridge
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from lazypredict.Supervised import LazyRegressor
from flaml import AutoML

# Darts imports
from darts import TimeSeries as DartsSeries
from darts.models import (ExponentialSmoothing, AutoARIMA as DartsAutoARIMA,
                           NaiveSeasonal, NaiveDrift)
from darts.metrics import mae as darts_mae, rmse as darts_rmse, mape as darts_mape

from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

pd.set_option("display.max_columns", 40)
plt.rcParams["figure.figsize"] = (14, 5)
sns.set_style("whitegrid")
print("Imports OK")

## Configuration & Constants

In [ ]:
PROJECT      = "Energy Load Forecasting"
KAGGLE_SLUG  = "nicholasjhana/energy-consumption-generation-prices-and-weather"
TARGET       = "total load actual"
TIME_COL     = "time"
FREQ         = "h"
SEASON_DAY   = 24   # daily seasonality
SEASON_WEEK  = 168  # weekly seasonality
HORIZON      = 168  # forecast 1 week
TEST_SIZE    = HORIZON
VAL_SIZE     = HORIZON
RANDOM_STATE = 42
FLAML_BUDGET = 120  # seconds

DATA_DIR = pathlib.Path("data"); DATA_DIR.mkdir(exist_ok=True)
print(f"Horizon: {HORIZON}h | Test set: {TEST_SIZE}h | Season: {SEASON_DAY}h / {SEASON_WEEK}h")

## Kaggle Credential Check

In [ ]:
import os, pathlib

cred_found = (os.environ.get("KAGGLE_USERNAME") or
              os.environ.get("KAGGLE_KEY") or
              os.environ.get("KAGGLE_API_TOKEN") or
              (pathlib.Path.home() / ".kaggle" / "kaggle.json").exists())

if cred_found:
    print("Kaggle credentials found. Ready to download.")
else:
    print("="*55)
    print("WARNING: Kaggle credentials not found.")
    print("="*55)
    print("Option 1 — kaggle.json file:")
    print("  1. Visit https://www.kaggle.com/settings")
    print("  2. Click 'Create New Token' → kaggle.json downloaded")
    print("  3. Move it to ~/.kaggle/kaggle.json")
    print()
    print("Option 2 — environment variables:")
    print("  set KAGGLE_USERNAME=your_username")
    print("  set KAGGLE_KEY=your_api_key")
    print()
    print("Then re-run this cell and the download cell below.")

## Dataset Download & Loading

kagglehub caches locally — re-running this cell will not re-download if the data already exists.

In [ ]:
import kagglehub

try:
    data_path = pathlib.Path(kagglehub.dataset_download(KAGGLE_SLUG))
    print(f"Data path: {data_path}")
except Exception as e:
    print(f"kagglehub failed: {e}")
    print("Trying kaggle CLI fallback...")
    os.system(f"kaggle datasets download -d {KAGGLE_SLUG} -p data/ --unzip")
    data_path = pathlib.Path("data")

csv_files = list(data_path.rglob("*.csv"))
print(f"Found {len(csv_files)} CSV(s):")
for f in csv_files:
    print(f"  {f.name}  ({f.stat().st_size/1e6:.1f} MB)")

### Load & Initial Inspection

The main file is nergy_dataset.csv (energy generation, load, prices) and optionally weather_features.csv.

In [ ]:
# Identify the main energy file
energy_file = next((f for f in csv_files if "energy_dataset" in f.name), csv_files[0])
print(f"Loading: {energy_file.name}")

df_raw = pd.read_csv(energy_file)
print(f"Shape: {df_raw.shape}")
print(f"Columns: {list(df_raw.columns)}")
df_raw.head(3)

## Data Validation Checks

We verify column names, missing values, duplicates, and parse the timestamp column.

In [ ]:
print("DATA VALIDATION REPORT")
print("="*50)

# 1. Timestamp column
assert TIME_COL in df_raw.columns, f"'{TIME_COL}' not found! Available: {list(df_raw.columns)}"
df_raw[TIME_COL] = pd.to_datetime(df_raw[TIME_COL], utc=True)
df_raw[TIME_COL] = df_raw[TIME_COL].dt.tz_convert("Europe/Madrid").dt.tz_localize(None)
print(f"Date range: {df_raw[TIME_COL].min()}  →  {df_raw[TIME_COL].max()}")
print(f"Total rows: {len(df_raw):,}")

# 2. Target column
if TARGET in df_raw.columns:
    print(f"Target '{TARGET}': {df_raw[TARGET].notna().sum():,} non-null values")
else:
    # Try case-insensitive match
    match = next((c for c in df_raw.columns if c.lower().replace(" ","_") == TARGET.lower().replace(" ","_")), None)
    print(f"Warning: '{TARGET}' not found directly. Closest: {match}")

# 3. Missing values in target
miss = df_raw[TARGET].isna().sum()
print(f"Missing in target: {miss} ({miss/len(df_raw)*100:.2f}%)")

# 4. Duplicates
dupes = df_raw[TIME_COL].duplicated().sum()
print(f"Duplicate timestamps: {dupes}")

# 5. Numeric sanity
print(f"Target range: {df_raw[TARGET].min():.0f} MW  →  {df_raw[TARGET].max():.0f} MW")
print(f"Zeros in target: {(df_raw[TARGET]==0).sum()}")
print(f"Negative in target: {(df_raw[TARGET]<0).sum()}")

## Exploratory Data Analysis

### Full time-series view

In [ ]:
# Build clean univariate series
ts_df = (df_raw[[TIME_COL, TARGET]]
         .drop_duplicates(TIME_COL)
         .sort_values(TIME_COL)
         .rename(columns={TIME_COL: "ds", TARGET: "y"})
         .dropna())

print(f"Clean series: {len(ts_df):,} hourly observations")
print(ts_df["y"].describe())

fig = px.line(ts_df, x="ds", y="y",
              title="Spain Hourly Electricity Load (MW) — Full Series",
              labels={"y":"Load (MW)", "ds":"Date"})
fig.update_layout(template="plotly_white")
fig.show()

### Multi-scale Seasonality

Energy load has two dominant seasonal cycles: **daily (24h)** and **weekly (168h)**.

In [ ]:
# Average load profile by hour-of-day and day-of-week
ts_df["hour"] = ts_df["ds"].dt.hour
ts_df["dow"]  = ts_df["ds"].dt.dayofweek
ts_df["month"] = ts_df["ds"].dt.month

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

ts_df.groupby("hour")["y"].mean().plot(ax=axes[0], marker="o", color="steelblue")
axes[0].set_title("Average Load by Hour of Day")
axes[0].set_xlabel("Hour"); axes[0].set_ylabel("Mean Load (MW)")

dow_means = ts_df.groupby("dow")["y"].mean()
dow_means.index = ["Mon","Tue","Wed","Thu","Fri","Sat","Sun"]
dow_means.plot(kind="bar", ax=axes[1], color="coral", edgecolor="black")
axes[1].set_title("Average Load by Day of Week")
axes[1].set_ylabel("Mean Load (MW)"); axes[1].tick_params(axis="x", rotation=45)

plt.tight_layout(); plt.show()

In [ ]:
# Monthly seasonality
monthly = ts_df.groupby("month")["y"].mean()
monthly.index = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]
fig = px.bar(x=monthly.index, y=monthly.values,
             title="Average Load by Month (Annual seasonality)",
             labels={"x":"Month","y":"Mean Load (MW)"}, color=monthly.values,
             color_continuous_scale="RdYlBu_r")
fig.update_layout(template="plotly_white", showlegend=False)
fig.show()

### Seasonal Decomposition (Weekly Period)

In [ ]:
# Use a 2-week window for decomposition
sample = ts_df.set_index("ds")["y"].asfreq("h").interpolate()
n_weeks = len(sample)//(SEASON_WEEK*4)
sample_trim = sample.iloc[:n_weeks*SEASON_WEEK*4]

if len(sample_trim) > 2*SEASON_WEEK:
    decomp = seasonal_decompose(sample_trim, model="additive", period=SEASON_WEEK)
    fig, ax = plt.subplots(4,1, figsize=(14,12), sharex=True)
    decomp.observed.plot(ax=ax[0], title="Observed")
    decomp.trend.plot(ax=ax[1], title="Trend")
    decomp.seasonal.plot(ax=ax[2], title="Seasonal (weekly)")
    decomp.resid.plot(ax=ax[3], title="Residual")
    plt.tight_layout(); plt.show()
else:
    print("Not enough data for weekly decomposition.")

In [ ]:
# ACF / PACF at daily lags
fig, axes = plt.subplots(1,2, figsize=(14,5))
plot_acf(ts_df["y"].dropna(), lags=72, ax=axes[0],
         title="ACF — first 72 lags (3 days)")
plot_pacf(ts_df["y"].dropna(), lags=72, ax=axes[1],
          title="PACF — first 72 lags (3 days)")
plt.tight_layout(); plt.show()

In [ ]:
# Stationarity test
adf = adfuller(ts_df["y"].dropna())
print("Augmented Dickey-Fuller Test")
print(f"  Statistic : {adf[0]:.4f}")
print(f"  p-value   : {adf[1]:.4f}")
for k,v in adf[4].items(): print(f"  Critical ({k}): {v:.4f}")
print()
print("Series is STATIONARY" if adf[1]<0.05 else "Series may be NON-STATIONARY — check for unit root")

## Target Analysis

Energy load has a characteristic double-peak pattern on weekdays (morning ramp ~8h, evening peak ~20h) and a single lower peak on weekends.

In [ ]:
from scipy.stats import zscore

print("Target statistics:")
print(ts_df["y"].describe().round(2))

# Z-score outlier detection
z = np.abs(zscore(ts_df["y"].dropna()))
outlier_thresh = 3.5
outliers = ts_df[z > outlier_thresh]
print(f"
Outliers (|z|>{outlier_thresh}): {len(outliers)} rows ({len(outliers)/len(ts_df)*100:.2f}%)")
if len(outliers) > 0:
    print(outliers[["ds","y"]].head(5).to_string(index=False))

## Train / Validation / Test Split

We use the **last 168 hours (1 week) as test**, the preceding 168 as validation, and everything before as training.

No shuffling — temporal order is sacred in time series.

In [ ]:
n = len(ts_df)
ts_test  = ts_df.iloc[n-TEST_SIZE:].copy()
ts_val   = ts_df.iloc[n-TEST_SIZE-VAL_SIZE:n-TEST_SIZE].copy()
ts_train = ts_df.iloc[:n-TEST_SIZE-VAL_SIZE].copy()
ts_trainval = ts_df.iloc[:n-TEST_SIZE].copy()

print(f"Train:  {len(ts_train):,} h  {ts_train['ds'].min().date()} → {ts_train['ds'].max().date()}")
print(f"Val:    {len(ts_val):,} h  {ts_val['ds'].min().date()} → {ts_val['ds'].max().date()}")
print(f"Test:   {len(ts_test):,} h  {ts_test['ds'].min().date()} → {ts_test['ds'].max().date()}")

assert ts_train["ds"].max() < ts_val["ds"].min(), "Leak: train/val overlap"
assert ts_val["ds"].max() < ts_test["ds"].min(), "Leak: val/test overlap"
print("Split is leak-free.")

fig = go.Figure()
fig.add_trace(go.Scatter(x=ts_train["ds"], y=ts_train["y"], name="Train", line=dict(color="steelblue")))
fig.add_trace(go.Scatter(x=ts_val["ds"],   y=ts_val["y"],   name="Val",   line=dict(color="orange")))
fig.add_trace(go.Scatter(x=ts_test["ds"],  y=ts_test["y"],  name="Test",  line=dict(color="crimson")))
fig.update_layout(title="Train/Val/Test Split", template="plotly_white",
                  xaxis_title="Date", yaxis_title="Load (MW)")
fig.show()

## Preprocessing

1. Fill rare missing hours via linear interpolation (< 0.1% of observations)
2. Cap extreme outliers at ±4 standard deviations
3. No scaling for tree-based models; scaling will be applied inside Darts models as needed

In [ ]:
def preprocess(df_in):
    df = df_in.copy().sort_values("ds").reset_index(drop=True)
    df = df.drop_duplicates("ds", keep="last")
    n_miss = df["y"].isna().sum()
    if n_miss > 0:
        df["y"] = df["y"].interpolate("linear")
        print(f"  Interpolated {n_miss} missing values")
    # Soft outlier cap
    mu, sigma = df["y"].mean(), df["y"].std()
    cap_lo, cap_hi = mu - 4*sigma, mu + 4*sigma
    n_cap = ((df["y"] < cap_lo) | (df["y"] > cap_hi)).sum()
    if n_cap > 0:
        df["y"] = df["y"].clip(cap_lo, cap_hi)
        print(f"  Capped {n_cap} extreme outliers")
    return df

ts_train    = preprocess(ts_train)
ts_val      = preprocess(ts_val)
ts_test     = preprocess(ts_test)
ts_trainval = preprocess(ts_trainval)
print("Preprocessing done.")

## Feature Engineering

For the tabular approaches (LazyPredict, FLAML) we create:
- **Lag features**: 1h, 2h, 3h, 24h (same hour yesterday), 168h (same hour last week)
- **Rolling statistics**: 24h mean and std
- **Calendar features**: hour, day-of-week, month, is_weekend, is_holiday

> Key rule: all features use .shift(1) or greater — no look-ahead.

In [ ]:
def make_lag_features(df_in):
    df = df_in.copy()
    for lag in [1, 2, 3, 24, 168]:
        df[f"lag_{lag}h"] = df["y"].shift(lag)
    for w in [6, 24]:
        df[f"roll_mean_{w}h"] = df["y"].shift(1).rolling(w).mean()
        df[f"roll_std_{w}h"]  = df["y"].shift(1).rolling(w).std()
    df["hour"]       = df["ds"].dt.hour
    df["dow"]        = df["ds"].dt.dayofweek
    df["month"]      = df["ds"].dt.month
    df["is_weekend"] = (df["dow"] >= 5).astype(int)
    df["hour_sin"]   = np.sin(2*np.pi*df["hour"]/24)
    df["hour_cos"]   = np.cos(2*np.pi*df["hour"]/24)
    df["dow_sin"]    = np.sin(2*np.pi*df["dow"]/7)
    df["dow_cos"]    = np.cos(2*np.pi*df["dow"]/7)
    return df

# Build on full series then re-split to preserve alignment
ts_all = pd.concat([ts_train, ts_val, ts_test]).reset_index(drop=True)
ts_feat = make_lag_features(ts_all)

feat_cols = [c for c in ts_feat.columns if c not in ["ds","y"]]
train_f = ts_feat.iloc[:len(ts_train)].dropna().copy()
val_f   = ts_feat.iloc[len(ts_train):len(ts_train)+len(ts_val)].dropna().copy()
test_f  = ts_feat.iloc[len(ts_train)+len(ts_val):].dropna().copy()

print(f"Feature count: {len(feat_cols)}")
print(f"Tabular sizes — train:{len(train_f)}, val:{len(val_f)}, test:{len(test_f)}")

## Baseline Approaches

Three naive baselines for energy load:
1. **Naive** — last known hour repeated
2. **Seasonal Naive (24h)** — same hour yesterday
3. **Seasonal Naive (168h)** — same hour last week ← typically strongest baseline

In [ ]:
results = []

def evaluate(actual, pred, name):
    a, p = np.array(actual).flatten(), np.array(pred).flatten()
    n = min(len(a), len(p))
    a, p = a[:n], p[:n]
    mae  = mean_absolute_error(a, p)
    rmse = np.sqrt(mean_squared_error(a, p))
    mask = a != 0
    mape = np.mean(np.abs((a[mask]-p[mask])/a[mask]))*100 if mask.sum()>0 else np.nan
    print(f"{name:<35s} MAE={mae:8.1f}  RMSE={rmse:8.1f}  MAPE={mape:5.2f}%")
    results.append({"model":name,"MAE":mae,"RMSE":rmse,"MAPE":mape})
    return p

# Naive (last value)
naive_val = np.full(len(ts_test), ts_trainval["y"].iloc[-1])
evaluate(ts_test["y"], naive_val, "Naive (last hour)")

# Seasonal naive 24h
sn24 = np.tile(ts_trainval["y"].iloc[-SEASON_DAY:].values,
               len(ts_test)//SEASON_DAY+1)[:len(ts_test)]
evaluate(ts_test["y"], sn24, "Seasonal Naive (24h — same hr yesterday)")

# Seasonal naive 168h  ← best naive for energy
sn168 = np.tile(ts_trainval["y"].iloc[-SEASON_WEEK:].values,
                len(ts_test)//SEASON_WEEK+1)[:len(ts_test)]
evaluate(ts_test["y"], sn168, "Seasonal Naive (168h — same hr last week)")

## LazyPredict Benchmark

We run LazyPredict on the lag-feature tabular view to screen 40+ regression algorithms quickly.

> Note: LazyPredict is a _tabular_ benchmark tool. It does not forecast natively — it learns a function (lag_features) → load. This is valid when features encode the temporal context correctly.

In [ ]:
X_tr = train_f[feat_cols]; y_tr = train_f["y"]
X_va = val_f[feat_cols];   y_va = val_f["y"]

print(f"LazyPredict — train:{X_tr.shape}, val:{X_va.shape}")
try:
    lazy = LazyRegressor(verbose=0, ignore_warnings=True)
    lazy_models, _ = lazy.fit(X_tr, X_va, y_tr, y_va)
    print(lazy_models.head(15).to_string())
    lazy_top5 = lazy_models.head(5).index.tolist()
    print(f"
Top-5 LazyPredict: {lazy_top5}")
except Exception as e:
    print(f"LazyPredict error: {e}")
    lazy_top5 = []

## FLAML AutoML

FLAML searches over model families (LightGBM, XGBoost, Random Forest, etc.) within a time budget and returns the best configuration.

In [ ]:
X_all = pd.concat([train_f, val_f])[feat_cols]
y_all = pd.concat([train_f, val_f])["y"]
X_te  = test_f[feat_cols]

flaml_model = AutoML()
flaml_model.fit(X_all, y_all, task="regression", metric="rmse",
                time_budget=FLAML_BUDGET, verbose=0, seed=RANDOM_STATE)

flaml_pred = flaml_model.predict(X_te)
print(f"Best FLAML estimator : {flaml_model.best_estimator}")
print(f"Best FLAML config    : {flaml_model.best_config}")
evaluate(test_f["y"], flaml_pred, f"FLAML ({flaml_model.best_estimator})")

## Darts — Dedicated Time-Series Library

**Why Darts for energy load?**

Darts provides a unified API for classical and neural forecasting models with native support for multiple seasonality periods. We fit three models:
1. **ExponentialSmoothing** — handles trend + seasonality elegantly
2. **AutoARIMA** — automatically selects ARIMA order
3. **NaiveSeasonal (168h)** — the strongest statistical baseline

All models operate on the raw hourly series — no feature engineering needed.

In [ ]:
# Convert to Darts TimeSeries
def to_darts(df):
    s = df.set_index("ds")["y"].asfreq("h").interpolate()
    return DartsSeries.from_series(s)

train_d  = to_darts(ts_trainval)
test_d   = to_darts(ts_test)
horizon  = len(ts_test)

darts_preds = {}

# 1. Exponential Smoothing
try:
    ets = ExponentialSmoothing(seasonal_periods=SEASON_WEEK, trend=True,
                                seasonal="add", damped=False)
    ets.fit(train_d)
    p = ets.predict(horizon)
    darts_preds["Darts-ETS"] = p
    evaluate(ts_test["y"].values, p.values().flatten(), "Darts-ETS")
except Exception as e:
    print(f"Darts ETS failed: {e}")

# 2. AutoARIMA
try:
    arima = DartsAutoARIMA()
    arima.fit(train_d)
    p = arima.predict(horizon)
    darts_preds["Darts-AutoARIMA"] = p
    evaluate(ts_test["y"].values, p.values().flatten(), "Darts-AutoARIMA")
except Exception as e:
    print(f"Darts AutoARIMA failed: {e}")

# 3. Seasonal Naive (168h — same-hour-last-week)
try:
    sn = NaiveSeasonal(K=SEASON_WEEK)
    sn.fit(train_d)
    p = sn.predict(horizon)
    darts_preds["Darts-SeasonalNaive-168h"] = p
    evaluate(ts_test["y"].values, p.values().flatten(), "Darts-SeasonalNaive-168h")
except Exception as e:
    print(f"Darts SeasonalNaive failed: {e}")

In [ ]:
# Plot Darts forecasts vs actual
fig = go.Figure()
fig.add_trace(go.Scatter(x=ts_test["ds"], y=ts_test["y"],
                          name="Actual", line=dict(color="black", width=2)))
colors = ["steelblue","darkorange","green"]
for (name, pred), color in zip(darts_preds.items(), colors):
    fig.add_trace(go.Scatter(x=ts_test["ds"], y=pred.values().flatten(),
                              name=name, line=dict(color=color, dash="dash")))
fig.update_layout(title="Darts Forecasts vs Actual Load",
                  xaxis_title="Date", yaxis_title="Load (MW)",
                  template="plotly_white")
fig.show()

## Top 3 Model Selection

Ranked by RMSE on the held-out test week.

In [ ]:
results_df = pd.DataFrame(results).sort_values("RMSE").reset_index(drop=True)
print("ALL RESULTS (ranked by RMSE)")
print("="*65)
print(results_df.to_string(index=False))
print()
top3 = results_df.head(3)
print("TOP 3 MODELS")
print("="*65)
print(top3.to_string(index=False))

fig = px.bar(results_df, x="model", y="RMSE",
             title="Model Comparison — RMSE (lower is better)",
             color="RMSE", color_continuous_scale="RdYlGn_r")
fig.update_layout(xaxis_tickangle=-40, template="plotly_white")
fig.show()

## Final Training & Evaluation of Top 3

In [ ]:
print("Top 3 for final evaluation:")
for _, row in top3.iterrows():
    print(f"  {row['model']}  →  RMSE={row['RMSE']:.1f}  MAE={row['MAE']:.1f}  MAPE={row['MAPE']:.2f}%")

# Overlay top 3 forecasts on test week
fig = go.Figure()
fig.add_trace(go.Scatter(x=ts_test["ds"], y=ts_test["y"],
                          name="Actual", line=dict(color="black", width=3)))

# FLAML prediction (already computed for test_f rows)
if len(test_f) > 0 and flaml_pred is not None:
    fig.add_trace(go.Scatter(x=test_f["ds"], y=flaml_pred,
                              name=f"FLAML ({flaml_model.best_estimator})",
                              line=dict(dash="dot")))

for name, pred in list(darts_preds.items())[:2]:
    fig.add_trace(go.Scatter(x=ts_test["ds"], y=pred.values().flatten(),
                              name=name, line=dict(dash="dash")))

fig.update_layout(title="Top 3 Models — Test Week Forecast",
                  xaxis_title="Date", yaxis_title="Load (MW)",
                  template="plotly_white")
fig.show()

## Error Analysis

Good forecasting insight comes from understanding **when** and **where** errors are largest — not just average metrics.

In [ ]:
if flaml_pred is not None and len(test_f) > 0:
    actual_arr = test_f["y"].values
    pred_arr   = flaml_pred[:len(actual_arr)]
    errors     = actual_arr - pred_arr
    abs_errors = np.abs(errors)

    print(f"Forecast Bias (mean error) : {errors.mean():+.1f} MW")
    print(f"MAE                        : {abs_errors.mean():.1f} MW")
    print(f"Max absolute error         : {abs_errors.max():.1f} MW")
    print(f"Hours with >5% error       : {(abs_errors/actual_arr*100 > 5).sum()} / {len(actual_arr)}")

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    axes[0].hist(errors, bins=30, color="steelblue", edgecolor="black", alpha=0.8)
    axes[0].axvline(0, color="red", lw=2, ls="--")
    axes[0].set_title("Error Distribution (MW)")

    axes[1].plot(range(len(errors)), errors, alpha=0.7, color="darkorange")
    axes[1].axhline(0, color="red", lw=1, ls="--")
    axes[1].set_title("Errors Over Forecast Horizon")
    axes[1].set_xlabel("Hour ahead")

    # Error by hour of day
    err_by_hour = pd.Series(abs_errors, index=test_f["ds"]).groupby(
        test_f.set_index("ds").index.hour).mean()
    axes[2].bar(err_by_hour.index, err_by_hour.values, color="coral", edgecolor="black")
    axes[2].set_title("Mean |Error| by Hour of Day")
    axes[2].set_xlabel("Hour")

    plt.tight_layout(); plt.show()

## Interpretation & Insights

### Key findings from this run
1. **Seasonal Naive (168h)** is a very strong baseline for energy load — same-hour-last-week captures most variance
2. **FLAML** typically wins by learning interaction effects between hour-of-day, day-of-week, and recent load history that classical models miss
3. **Darts ETS** handles the trend + weekly seasonality but can struggle with irregular demand spikes (holidays, extreme weather)
4. The **peak ramp hours** (6–9h and 17–20h) tend to have the highest absolute errors — these are the most operationally important hours to forecast accurately

### Domain interpretation
- MAPE < 2% is considered excellent for day-ahead load forecasting
- 2–4% is good (utility-grade)
- > 5% is problematic for real dispatch decisions

## Limitations

1. **Weather not used as exogenous variable**: Adding temperature, humidity, and wind speed as covariates would significantly improve accuracy (especially for cooling/heating load)
2. **No holiday calendar**: Public holidays create anomalous loads — a holiday indicator feature would reduce errors on those days
3. **Single-week test**: One test week may not be representative; rolling-origin CV over 52 weeks would give a more robust accuracy estimate
4. **DST transitions ignored**: Daylight saving time creates one 23h and one 25h day per year — not handled here
5. **Neural models excluded**: N-BEATS and TCN typically outperform classical models at short horizons on energy data, but require GPU for practical training

## How to Improve This Project

1. **Add weather regressors**: join the weather_features.csv (temperature, humidity) via the timestamp key
2. **Add holiday indicator**: use the holidays library for Spanish public and regional holidays
3. **Rolling-origin CV**: evaluate on 52 weekly windows to get a robust MAPE distribution
4. **N-BEATS model**: try darts.models.NBEATSModel for a neural alternative (requires more training time)
5. **Conformal prediction intervals**: wrap any model with darts.utils.statistics.ConformalNaivePredictor to get calibrated prediction intervals
6. **Multi-step loss**: train with direct multi-step loss instead of recursive 1-step, which often reduces error accumulation

## Production Considerations

A production energy load forecasting system would need:
1. **Automated hourly retrain** on newest data (or at minimum a daily retrain for the next 24h)
2. **Integration with the spot market timeline**: Day-ahead bids close at 12:00 CET — the model must produce forecasts before that deadline
3. **Ensemble/regression combination**: Blend model output with ENTSO-E's official forecast as a strong prior
4. **Alerting**: Trigger human review when the model's predicted load differs > 5% from the official forecast
5. **Auditability**: Log all inputs and outputs — grid dispatch decisions must be explainable to regulators

## Common Mistakes to Avoid

1. **Using 	otal load forecast as a feature WITHOUT lagging it** — this would be data leakage (the official forecast for time *t* is not known before *t−24h*)
2. **Scaling the target without inverse-transforming predictions** — MAPE will look artificially good but real MW errors will be wrong
3. **Random shuffle split** — destroys the time structure entirely
4. **Testing only on one month in summer** — load patterns differ dramatically by season; test across all seasons
5. **Forgetting DST** — a single DST transition can corrupt an entire model's output if timestamps are handled naively

## Mini Challenge / Exercises

1. **Add temperature as a covariate**: load weather_features.csv, join on dt_iso, and add Madrid_temp_k as a Darts future-covariates column. How much does MAPE improve?
2. **Implement a rolling-origin backtest**: write a loop that evaluates on 8 consecutive test weeks and reports the distribution of MAPE values
3. **Next-day peak prediction**: instead of forecasting every hour, predict only the daily peak MW. How does your model compare on this KPI?
4. **Decomposition residual model**: subtract a seasonal naive (168h) forecast, model the residuals with XGBoost lag features, then add them back. Does this two-stage approach beat the single-stage models?
5. **Compare ENTSO-E official forecast**: plot and compute RMSE for 	otal load forecast vs actual — how does your model compare to the professional grid-operator forecast?

## Final Summary & Key Takeaways

### What We Did
- Downloaded and validated the ENTSO-E Spain energy dataset (4 years, 35,000 hourly rows)
- Explored dual seasonality (24h daily + 168h weekly) with decomposition, ACF/PACF, and hour/DOW profiles
- Built a robust time-aware split (last 168h = test week)
- Engineered domain-specific lag features (lag-24h, lag-168h, rolling-24h stats, hour/DOW cyclical encoding)
- Benchmarked 40+ models with LazyPredict on lag features
- Ran FLAML AutoML for optimised regression
- Fitted Darts ExponentialSmoothing, AutoARIMA, and SeasonalNaive-168h
- Selected Top 3 by RMSE, analysed intra-day error patterns

### Key Takeaways
1. For energy load, the **168h seasonal naive is a deceptively strong baseline** — many complex models barely beat it
2. **Same-hour-last-week lag** (lag_168h) is the single most important feature for tabular approaches
3. **Hour-of-day and day-of-week are not optional** — load patterns are fundamentally structured around human schedules
4. **Error is highest at peak ramp hours** — this is where better models (with weather inputs) add the most value
5. MAPE of 2–4% on a 168h horizon is realistic without weather covariates; adding temperature can push this below 2%

---
*Educational notebook — part of the 50-project Time Series Forecasting portfolio.*